In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from tensorflow.keras.preprocessing.image import img_to_array
import matplotlib.pyplot as plt

# Load the pre-trained VGG16 model
model = tf.keras.models.load_model('vgg16_tree_classifier_augmented.h5')

# Load the SAM model
sam_checkpoint = "sam_vit_h_4b8939.pth"
device = "cuda" if tf.config.list_physical_devices('GPU') else "cpu"
sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
sam.to(device=device)

# Function to remove small masks based on area
def remove_small_masks(masks, min_area=10000):
    """Remove masks smaller than a given area."""
    filtered_masks = [mask for mask in masks if np.sum(mask['segmentation']) >= min_area]
    return filtered_masks

# Function to classify masked regions using VGG model
def classify_masked_regions(masked_image, model):
    """Classify masked image region using the VGG model."""
    masked_image_resized = cv2.resize(masked_image, (224, 224))
    masked_image_array = img_to_array(masked_image_resized)
    masked_image_array = np.expand_dims(masked_image_array, axis=0) / 255.0  # Normalize

    # Get prediction from the model
    prediction = model.predict(masked_image_array)
    return prediction[0][0] > 0.5  # Return True if classified as a tree
    
# Function to resolve overlapping masks by keeping the largest non-overlapping masks
def resolve_overlaps(masks):
    if not masks:
        return masks
    keep_masks = np.ones(len(masks), dtype=bool)
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if np.any(np.logical_and(masks[i]['segmentation'], masks[j]['segmentation'])):
                if np.sum(masks[i]['segmentation']) >= np.sum(masks[j]['segmentation']):
                    keep_masks[j] = False
                else:
                    keep_masks[i] = False
    return [masks[i] for i in range(len(masks)) if keep_masks[i]]

# Function to process an image and apply SAM masks
def process_image(image_path, sam, model, output_dir, min_mask_area):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_filename = os.path.splitext(os.path.basename(image_path))[0]

    # Generate masks
    mask_generator = SamAutomaticMaskGenerator(sam, min_mask_region_area=min_mask_area)
    masks = mask_generator.generate(image)

    # Remove small masks
    filtered_masks = remove_small_masks(masks, min_area=min_mask_area)

    # Create a copy of the original image for blending
    tree_mask_overlay = image.copy()

    # Create a separate image for the masks to ensure better visibility of the overlay
    tree_mask = np.zeros_like(image, dtype=np.uint8)
    tree_mask_count = 0  # Track the number of tree masks

    # Process each mask
    for mask in filtered_masks:
        mask_bool = mask['segmentation'].astype(bool)
        masked_image = np.zeros_like(image)

        # Apply the mask to the original image
        for c in range(3):  # Assuming original_image is color (3 channels)
            masked_image[:, :, c] = image[:, :, c] * mask_bool

        # Classify the masked image
        is_tree = classify_masked_regions(masked_image, model)

        # If it's classified as a tree, add the mask to the overlay and count it
        if is_tree:
            tree_mask_count += 1
            for c in range(3):  # Apply the mask to the tree overlay
                tree_mask[:, :, c] += mask_bool.astype(np.uint8) * 255  # Mark the mask in white

    # Apply the tree mask as a transparent overlay on the original image
    blended_image = cv2.addWeighted(image, 0.7, tree_mask, 0.3, 0)

    # Display the result
    plt.figure(figsize=(10, 10))
    plt.imshow(blended_image)
    plt.axis('off')
    plt.title(f"Tree Mask Overlay: {image_filename}")
    plt.show()

    # Print the number of tree masks
    print(f"Processed {image_path}. Number of tree masks classified: {tree_mask_count}")

def main():
    image_folder = '/content/drive/MyDrive/100FPLAN'
    output_dir = '/content/mask_outputs'
    min_mask_area = 10000  # Set minimum mask area

    for filename in os.listdir(image_folder):
        if filename.lower().endswith(".jpg"):
            image_path = os.path.join(image_folder, filename)
            process_image(image_path, sam, model, output_dir, min_mask_area)

if __name__ == "__main__":
    main()
